# RoboTwin × LingBot-VLA-v2 on ROCm

This notebook is the interactive entry point for the `external-data` image. Run cells selectively: model serving and evaluation are long-running GPU workloads. The platform-provided data must be available at `/models/robotwin-persistent`. Runtime outputs are written to `/workspace/runtime`.

In [ ]:
from pathlib import Path
import os, signal, subprocess, time, urllib.request

ROBOTWIN = Path('/RoboTwin')
RUNTIME = Path('/workspace/runtime')
PYTHON = '/opt/robotwin-env/bin/python'
# Official LingBot-VLA-v2 base checkpoint; do not use the RobotWin post-training model by default.
MODEL = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b'
os.environ.setdefault('AITER_TRITON_ONLY', '1')
os.environ.setdefault('FLASH_ATTENTION_TRITON_AMD_ENABLE', 'TRUE')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['TMPDIR'] = '/workspace/runtime/tmp'
os.environ['PYTHONPATH'] = '/opt/aiter' + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
RUNTIME.joinpath('outputs/logs').mkdir(parents=True, exist_ok=True)
RUNTIME.joinpath('tmp').mkdir(parents=True, exist_ok=True)
print('TMPDIR:', os.environ.get('TMPDIR'))
print('TORCHINDUCTOR_CACHE_DIR:', os.environ.get('TORCHINDUCTOR_CACHE_DIR'))
print('RoboTwin:', ROBOTWIN)
print('Model:', MODEL)

## 1. Environment and mounted-data check

In [ ]:
required = [
    ROBOTWIN / 'assets/objects/objaverse/list.json',
    ROBOTWIN / 'data/demo_clean',
    ROBOTWIN / 'data/lerobot',
    MODEL / 'model.safetensors.index.json',
]
for path in required:
    assert path.exists(), f'Missing: {path}'
subprocess.run([PYTHON, '-c', "import torch; print('torch=', torch.__version__, 'hip=', torch.version.hip, 'gpu=', torch.cuda.is_available(), 'count=', torch.cuda.device_count())"], check=True)

## 2. Start the model server on port 13400

Stop any existing process using port 13400 before running this cell. The server log is written under `/workspace/runtime/outputs/logs`.

In [ ]:
server_log = RUNTIME / 'outputs/logs/official_server.log'
server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('ab')
SERVER_PROCESS = subprocess.Popen(server_command, cwd=ROBOTWIN, stdout=server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('server pid:', SERVER_PROCESS.pid, 'log:', server_log)
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Server exited with code {SERVER_PROCESS.returncode}; inspect {server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Server did not become ready; inspect {server_log}')

## 3. Run 10 closed-loop `adjust_bottle` episodes

Evaluation disables video by default to reduce encoding and disk I/O. The two video switches are intentionally explicit: single-task evaluation uses `--additional_info eval_video_log=false`, while the benchmark uses `--no-video`. Do not simply delete these switches when videos are needed: the benchmark script defaults to no video, and the single-task default may vary with the evaluation entry point. To save MP4 files, change the single-task value to `eval_video_log=true` and replace `--no-video` with `--video` in each benchmark cell. This setting applies to the baseline, RoboTwin checkpoint, LoRA, and Full-SFT evaluations.

### 3.1 Validate the official base model with 10 closed-loop episodes

The official baseline checkpoint is primarily for validating model serving and the evaluation pipeline. Its RoboTwin success rate may be modest, so a low result here does not by itself indicate a broken deployment.

In [ ]:
EVAL_EPISODES = 10
# Video is disabled by default to reduce encoding time and disk I/O.
# To save single-task episode videos, change this to True; do not delete the
# eval_video_log argument below because its default depends on the entry point.
EVAL_VIDEO_LOG = False
# The benchmark script also defaults to no video. Change this to True to make
# every benchmark command use --video instead of --no-video.
BENCHMARK_VIDEO = False
eval_command = [
    PYTHON, str(ROBOTWIN / 'scripts/eval_policy_xpolicylab.py'),
    '--task_name', 'adjust_bottle', '--task_config', 'demo_clean',
    '--policy_name', 'LingBot-VLA-v2', '--protocol', 'lingbot_vla_v2',
    '--host', '127.0.0.1', '--port', '13400', '--device_id', '0',
    '--seed', '0', '--test_num', str(EVAL_EPISODES), '--expert_check', 'true',
    '--accept_expert_info_on_failure', 'true',
    '--eval_batch', 'false',
    '--additional_info', f'eval_video_log={str(EVAL_VIDEO_LOG).lower()}',
]
eval_env = os.environ.copy()
eval_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1')
benchmark_env = os.environ.copy()
benchmark_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1')
base_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
base_eval_elapsed = time.perf_counter() - base_eval_started
print(f'base evaluation total: {base_eval_elapsed:.2f}s ({base_eval_elapsed / 60:.2f} min)')
print(f'base evaluation average per episode: {base_eval_elapsed / EVAL_EPISODES:.2f}s')

The baseline evaluation above is only a deployment reference. If you need a stronger RoboTwin task result, evaluate the LingBot official RoboTwin checkpoint below; it has already been trained for RoboTwin and is expected to achieve a higher success rate.

The official baseline checkpoint is mainly for validating deployment, and its RoboTwin task performance may be modest. The LingBot official RoboTwin checkpoint used below has already been trained for RoboTwin and is expected to perform better on RoboTwin tasks, so it is the appropriate checkpoint for effect validation. It is provided by [Hugging Face](https://huggingface.co/robbyant/lingbot-vla-v2-6b-robotwin); see also the [LingBot-VLA-v2 repository](https://github.com/robbyant/lingbot-vla-v2). The competition/reproduction training must start from the baseline checkpoint in Section 2 and must not start from this already-trained RoboTwin checkpoint.

The following code uses the same `adjust_bottle` closed-loop settings as Section 3.1.

In [ ]:
ROBOTWIN_CHECKPOINT_ROOT = Path('/models/robotwin-persistent/models/robbyant_lingbot-vla-v2-6b-robotwin').expanduser().resolve()
ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT
if not (ROBOTWIN_CHECKPOINT / 'model.safetensors.index.json').exists():
    ROBOTWIN_CHECKPOINT = ROBOTWIN_CHECKPOINT_ROOT / 'checkpoints/global_step_50000/hf_ckpt'
ROBOTWIN_CHECKPOINT_CONFIG = ROBOTWIN_CHECKPOINT_ROOT / 'lingbotvla_cli.yaml'
ROBOTWIN_CHECKPOINT_EPISODES = EVAL_EPISODES
ROBOTWIN_CHECKPOINT_VIDEO = EVAL_VIDEO_LOG
assert ROBOTWIN_CHECKPOINT.exists(), f'Missing official RoboTwin checkpoint: {ROBOTWIN_CHECKPOINT}'
assert (ROBOTWIN_CHECKPOINT / 'model.safetensors.index.json').exists(), f'Invalid official RoboTwin checkpoint: {ROBOTWIN_CHECKPOINT}'
if True:  # Run the fixed official RoboTwin checkpoint evaluation.
    if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
        SERVER_PROCESS.wait(timeout=30)
    checkpoint_server_log = RUNTIME / 'outputs/logs/robotwin_checkpoint_server.log'
    checkpoint_server_command = ['bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'), '0', '13400', str(checkpoint_server_log), 'False', str(ROBOTWIN_CHECKPOINT)]
    checkpoint_env = os.environ.copy()
    checkpoint_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl', PYTHONUNBUFFERED='1')
    if ROBOTWIN_CHECKPOINT_CONFIG is not None:
        checkpoint_env['LINGBOTVLA_TRAINING_CONFIG'] = str(Path(ROBOTWIN_CHECKPOINT_CONFIG).expanduser().resolve())
    checkpoint_server_handle = checkpoint_server_log.open('ab')
    ROBOTWIN_CHECKPOINT_SERVER_PROCESS = subprocess.Popen(checkpoint_server_command, cwd=ROBOTWIN, env=checkpoint_env, stdout=checkpoint_server_handle, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(600):
        if ROBOTWIN_CHECKPOINT_SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f'Checkpoint server exited; inspect {checkpoint_server_log}')
        try:
            urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
            break
        except Exception:
            time.sleep(2)
    checkpoint_eval_command = [
        PYTHON, str(ROBOTWIN / 'scripts/eval_policy_xpolicylab.py'),
        '--task_name', 'adjust_bottle', '--task_config', 'demo_clean',
        '--policy_name', 'LingBot-VLA-v2', '--protocol', 'lingbot_vla_v2',
        '--host', '127.0.0.1', '--port', '13400', '--device_id', '0',
        '--seed', '0', '--test_num', str(ROBOTWIN_CHECKPOINT_EPISODES), '--expert_check', 'true',
        '--accept_expert_info_on_failure', 'true',
        '--eval_batch', 'false',
        '--additional_info', f'eval_video_log={str(ROBOTWIN_CHECKPOINT_VIDEO).lower()}',
    ]
    subprocess.run(checkpoint_eval_command, cwd=ROBOTWIN, env=checkpoint_env, check=True)
print('Evaluating the official LingBot RoboTwin checkpoint:', ROBOTWIN_CHECKPOINT)

### 3.2 Optional: clean + randomized, 100 tasks × 10 episodes, on 4/8 GPUs

This is the full 1,000-episode closed-loop benchmark, not a smoke test. It is disabled by default to avoid accidentally starting a multi-hour run; set `RUN_OFFICIAL_FULL_BENCHMARK = True` to enable it. `BENCHMARK_GPU_COUNT` supports `4` or `8` and defaults to `8`. One model server and one RoboTwin evaluation worker are launched per GPU; a shared task queue dynamically assigns the next unfinished task to the first free GPU. The cell stops the single server from Section 2 before starting the benchmark and stops all benchmark servers when it finishes.

Measured wall-clock timings (success-rate details are intentionally omitted here):

| Machine | Config | Completion | Wall time |
|---|---|---:|---:|
| 4 x W7900 | clean | 50 tasks / 500 episodes | 8 h 26 min 41 s |
| 4 x W7900 | randomized | 50 tasks / 500 episodes | 10 h 29 min 07 s |
| 4 x W7900 | total | 100 tasks / 1,000 episodes | 18 h 55 min 48 s |
| 8 x W7900 | clean | 50 tasks / 500 episodes | 6 h 34 min 23 s |
| 8 x W7900 | randomized | 50 tasks / 500 episodes | 8 h 37 min 58 s |
| 8 x W7900 | total | 100 tasks / 1,000 episodes | 15 h 12 min 21 s |

The estimates assume one model replica per GPU, the same model/checkpoint, `demo_clean`, 50 episodes, and broadly similar server/video settings. They are not linear model-throughput benchmarks: task lengths differ substantially, so the final long task determines the tail. The run writes one log per task plus `task_times.tsv`, `events.log`, and completion markers under `/workspace/runtime/outputs/<BENCHMARK_RUN_NAME>`. Set a new run name for an independent rerun.

In [ ]:
RUN_OFFICIAL_FULL_BENCHMARK = False
BENCHMARK_GPU_COUNT = 8  # Notebook default; supported values: 4 or 8
BENCHMARK_EPISODES = 10
BENCHMARK_RUN_NAME = f'both100x10_{BENCHMARK_GPU_COUNT}gpu'
BENCHMARK_RESUME = True

if RUN_OFFICIAL_FULL_BENCHMARK:
    if BENCHMARK_GPU_COUNT not in (4, 8):
        raise ValueError('BENCHMARK_GPU_COUNT must be 4 or 8')

    # The reusable benchmark script launches one model server per GPU.
    if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
        SERVER_PROCESS.wait(timeout=30)
        if 'server_log_handle' in globals() and not server_log_handle.closed:
            server_log_handle.close()
        print('single model server stopped before the multi-GPU benchmark')

    benchmark_script = (
        ROBOTWIN
        / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    )
    benchmark_command = [
        PYTHON, str(benchmark_script),
        '--gpu-count', str(BENCHMARK_GPU_COUNT),
        '--episodes', str(BENCHMARK_EPISODES),
        '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
    ]
    if BENCHMARK_RESUME:
        benchmark_command.append('--resume')

    subprocess.run(benchmark_command, cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('Skipped the long official-model 100-task benchmark')


The long benchmark above also uses the official baseline by default. For a stronger RoboTwin-oriented comparison, the aligned cell below runs the same multi-GPU benchmark with the official LingBot RoboTwin checkpoint. This checkpoint is for evaluation only; competition training must still start from the baseline checkpoint.

In [ ]:
RUN_ROBOTWIN_FULL_BENCHMARK = False
ROBOTWIN_BENCHMARK_CHECKPOINT = ROBOTWIN_CHECKPOINT
ROBOTWIN_BENCHMARK_CONFIG = ROBOTWIN_CHECKPOINT_CONFIG
ROBOTWIN_BENCHMARK_RUN_NAME = f'robotwin_checkpoint_both100x10_{BENCHMARK_GPU_COUNT}gpu'
if RUN_ROBOTWIN_FULL_BENCHMARK:
    robotwin_benchmark_script = (
        ROBOTWIN
        / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    )
    assert ROBOTWIN_BENCHMARK_CHECKPOINT.exists(), f'Missing official RoboTwin checkpoint: {ROBOTWIN_BENCHMARK_CHECKPOINT}'
    assert (ROBOTWIN_BENCHMARK_CHECKPOINT / 'model.safetensors.index.json').exists(), f'Invalid official RoboTwin checkpoint: {ROBOTWIN_BENCHMARK_CHECKPOINT}'
    robotwin_benchmark_env = benchmark_env.copy()
    robotwin_benchmark_env['LINGBOTVLA_TRAINING_CONFIG'] = str(ROBOTWIN_BENCHMARK_CONFIG)
    robotwin_benchmark_command = [
        PYTHON, str(robotwin_benchmark_script),
        '--gpu-count', str(BENCHMARK_GPU_COUNT),
        '--episodes', str(BENCHMARK_EPISODES),
        '--model-path', str(ROBOTWIN_BENCHMARK_CHECKPOINT),
        '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', ROBOTWIN_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
    ]
    if BENCHMARK_RESUME:
        robotwin_benchmark_command.append('--resume')
    subprocess.run(robotwin_benchmark_command, cwd=ROBOTWIN, env=robotwin_benchmark_env, check=True)
else:
    print('Skipped the long official-RoboTwin-checkpoint benchmark')

## 4. LoRA training, checkpoint merge, and inference

Choose `GPU_COUNT` as 4 or 8 and set `TRAIN_STEPS` explicitly; the notebook defaults to 8 GPUs. The default 100 steps is a smoke test for the complete pipeline, not an expectation of useful fine-tuning quality. Start with 1,000–5,000 steps for an experiment and select the final value from held-out closed-loop evaluation. The effective global batch size is 4 on four GPUs and 8 on eight GPUs. This cell stops the model server started above before training so it releases GPU memory.

### 4.1 Train the LoRA adapter

In [ ]:
if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
    SERVER_PROCESS.wait(timeout=30)
    if 'server_log_handle' in globals() and not server_log_handle.closed:
        server_log_handle.close()
    print('model server stopped before training')

GPU_COUNT = 8  # Notebook default; supported values: 4 or 8
TRAIN_STEPS = 100  # Smoke test; increase to e.g. 1000 or 5000 for fine-tuning
if GPU_COUNT not in (4, 8):
    raise ValueError('GPU_COUNT must be 4 or 8')
if TRAIN_STEPS <= 0:
    raise ValueError('TRAIN_STEPS must be positive')
lora_global_batch_size = max(4, GPU_COUNT)
gradient_accumulation_steps = lora_global_batch_size // GPU_COUNT
training_root = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
source_root = training_root / 'source/lingbot-vla-v2'
training_yaml = training_root / 'training/lingbotvla_cli.yaml'
training_output = RUNTIME / f'outputs/lora_{TRAIN_STEPS}steps_{GPU_COUNT}gpu'
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.gradient_accumulation_steps', str(gradient_accumulation_steps),
    '--train.global_batch_size', str(lora_global_batch_size),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
training_started = time.perf_counter()
subprocess.run(train_command, cwd=source_root, env=train_env, check=True)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training total: {training_elapsed:.2f}s ({training_elapsed / 60:.2f} min, {training_elapsed / 3600:.2f} h)')
print(f'LoRA wall-clock average per optimizer step: {training_elapsed / TRAIN_STEPS:.2f}s')

### 4.2 Merge the LoRA checkpoint

In [ ]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.exists(), f'Missing checkpoint: {checkpoint}'
merge_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint),
    '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
subprocess.run(merge_command, cwd=ROBOTWIN, check=True)
assert (MERGED / 'model.safetensors.index.json').exists(), f'Merge output is incomplete: {MERGED}'
print('merged model:', MERGED)

### 4.3 Start the merged LoRA model on port 13400

The original server was stopped before training, so the merged model reuses the same endpoint.

In [ ]:
merged_log = RUNTIME / 'outputs/logs/merged_server.log'
merged_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('ab')
MERGED_SERVER_PROCESS = subprocess.Popen(merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('merged server pid:', MERGED_SERVER_PROCESS.pid, 'log:', merged_log)
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited with code {MERGED_SERVER_PROCESS.returncode}; inspect {merged_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('merged server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')

### 4.4 Validate the merged LoRA model with 10 closed-loop episodes

In [ ]:
merged_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
print(f'merged evaluation total: {merged_eval_elapsed:.2f}s ({merged_eval_elapsed / 60:.2f} min)')
print(f'merged evaluation average per episode: {merged_eval_elapsed / EVAL_EPISODES:.2f}s')

### 4.5 Optional: run 100 tasks × 10 episodes against the merged LoRA model

This is a long 1,000-episode evaluation. Set `LORA_BENCHMARK_GPU_COUNT` to `4` or `8`; the following code cell passes that value to the unified benchmark script. The measured clean/randomized wall-clock table is in the Robotwin evaluation section of the Guide. The shared script starts one model server per GPU, so it first stops the merged-model server above. `--resume` skips tasks that already have a completion marker in the same run directory. Change `LORA_BENCHMARK_RUN_NAME` when you want a completely independent rerun instead of resuming existing results.


In [ ]:
RUN_LORA_FULL_BENCHMARK = False
LORA_BENCHMARK_GPU_COUNT = 8  # Notebook default; supported values: 4 or 8
LORA_BENCHMARK_RUN_NAME = f'lora_{TRAIN_STEPS}steps_both100x10_{LORA_BENCHMARK_GPU_COUNT}gpu'

if RUN_LORA_FULL_BENCHMARK:
    if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
        MERGED_SERVER_PROCESS.wait(timeout=30)
        if 'merged_log_handle' in globals() and not merged_log_handle.closed:
            merged_log_handle.close()
        print('single merged-model server stopped before the full benchmark')
    benchmark_script = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    subprocess.run([
        PYTHON, str(benchmark_script),
        '--gpu-count', str(LORA_BENCHMARK_GPU_COUNT),
        '--episodes', '10',
        '--model-path', str(MERGED),
        '--expert-check',
        '--accept-expert-info-on-failure',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--run-name', LORA_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
        '--resume',
    ], cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('Skipped the long merged-LoRA 100-task benchmark')


### 4.6 Optional: stop the merged LoRA model server

In [ ]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    if 'merged_log_handle' in globals() and not merged_log_handle.closed:
        merged_log_handle.close()
    print('merged model server stopped')
else:
    print('no merged server started by this notebook kernel')

## 5. Full-parameter SFT, checkpoint conversion, and inference

This is independent from the LoRA workflow above: it sets `use_lora=false` and updates all trainable model parameters. The default reproduction path enables the complete depth/video teacher; a no-teacher SFT run is available only as an optional comparison and is not recommended because its supervision and final quality are worse. `FULL_SFT_GPU_COUNT` supports 4 or 8 and defaults to 8. The reproduction configuration uses global batch 256; the launcher derives gradient accumulation as `global batch / (GPU count × micro batch)`. The default smoke test runs 10 steps; change `FULL_SFT_STEPS` and `FULL_SFT_SAVE_STEPS` as needed.

Full-teacher checkpoints contain extra alignment heads used only for depth/video loss. The deployment patch filters those training-only weights while retaining strict validation for all inference weights, so the converted checkpoint can be served by the action policy.

Keep `FULL_SFT_OPTIMIZER='adamw'` for both four- and eight-GPU reproduction. The validated global-batch-256 capacity points are micro batch 16 with accumulation 4 on four W7900 GPUs, and micro batch 16 with accumulation 2 on eight W7900 GPUs.

Use the four- and eight-GPU commands below for the reproducible AdamW runs. Adjust `MAX_STEPS` and `SAVE_STEPS` for the intended training duration and checkpoint frequency.

Under FSDP2, `FULL_SFT_ENABLE_FULL_SHARD=True` means `reshard_after_forward=True`: each wrapped module is sharded again after forward, lowering peak VRAM at the cost of another all-gather before backward. `False` keeps that module's full parameters through backward, which may be faster but generally uses more peak VRAM. It does not disable FSDP or replicate the entire model on every GPU. Keep `True` unless a one-step comparison proves that `False` fits safely.

The available full-SFT configurations are: (1) the default full-shard run below; (2) the same run with no reshard for the communication-versus-memory comparison; and (3) a full-shard run with `enable_fp32` and `use_future_image` enabled. They are alternatives, not consecutive training stages.

In [ ]:
# Radeon Cloud Global currently provides only 100 GB of persistent /workspace storage.
# Check both the larger ephemeral root filesystem and the persistent workspace before SFT.
subprocess.run(['df', '-h', '/', '/workspace'], check=True)
subprocess.run(['du', '-d1', '-h', str(RUNTIME / 'outputs')], check=True)

# Optional cleanup. This is disabled by default; set it to True only after reviewing the list.
DELETE_PREVIOUS_LORA_OUTPUTS = False
lora_output_candidates = sorted(set((RUNTIME / 'outputs').glob('lora_*')) | set((RUNTIME / 'outputs').glob('reproduction_*')))
if DELETE_PREVIOUS_LORA_OUTPUTS:
    print('The following previous LoRA output directories will be deleted:')
    for path in lora_output_candidates:
        print(' -', path)
    import shutil
    for path in lora_output_candidates:
        if path.is_dir():
            shutil.rmtree(path)
    print('Previous LoRA outputs deleted.')
else:
    print('No LoRA outputs were deleted. Set DELETE_PREVIOUS_LORA_OUTPUTS=True after reviewing the list.')

print('Reminder: intermediate files may use the larger / filesystem, but copy the final checkpoint, logs, and results to /workspace before the instance is destroyed.')

### 5.1 Run full-parameter SFT

In [ ]:
# GPU count may be 4 or 8; the notebook default is 8.
FULL_SFT_GPU_COUNT = 8
FULL_SFT_STEPS = 10  # Change to the desired training length
FULL_SFT_SAVE_STEPS = FULL_SFT_STEPS
FULL_SFT_TEACHER_MODE = 'full'  # Set to 'none' only for the optional no-teacher comparison
FULL_SFT_TIMEOUT_SECONDS = 0  # Let the 10-step smoke test finish naturally
FULL_SFT_GLOBAL_BATCH_SIZE = 256
# W7900 capacity-tested default; override when running a different platform.
FULL_SFT_MICRO_BATCH_SIZE = 16  # Four- and eight-GPU W7900 default
FULL_SFT_ENABLE_FULL_SHARD = True  # Recommended lower-memory default
FULL_SFT_OPTIMIZER = 'adamw'  # Default for the four- and eight-GPU W7900 runs
assert FULL_SFT_TEACHER_MODE in ('full', 'current_depth', 'none')
assert FULL_SFT_GPU_COUNT in (4, 8)
FULL_SFT_OUTPUT = RUNTIME / f'outputs/full_sft_{FULL_SFT_TEACHER_MODE}_{FULL_SFT_GPU_COUNT}gpu_{FULL_SFT_STEPS}steps'

for process_name in ('SERVER_PROCESS', 'MERGED_SERVER_PROCESS'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)

gpu_count = int(subprocess.check_output(
    [PYTHON, '-c', 'import torch; print(torch.cuda.device_count())'], text=True
).strip())
assert FULL_SFT_GPU_COUNT in (4, 8), 'FULL_SFT_GPU_COUNT must be 4 or 8'
assert gpu_count >= FULL_SFT_GPU_COUNT, f'Full SFT requested {FULL_SFT_GPU_COUNT} GPUs, found {gpu_count}'

full_sft_env = os.environ.copy()
full_sft_env.update({
    'GPU_COUNT': str(FULL_SFT_GPU_COUNT),
    'MAX_STEPS': str(FULL_SFT_STEPS),
    'SAVE_STEPS': str(FULL_SFT_SAVE_STEPS),
    'TEACHER_MODE': FULL_SFT_TEACHER_MODE,
    'TIMEOUT_SECONDS': str(FULL_SFT_TIMEOUT_SECONDS),
    'GLOBAL_BATCH_SIZE': str(FULL_SFT_GLOBAL_BATCH_SIZE),
    'MICRO_BATCH_SIZE': str(FULL_SFT_MICRO_BATCH_SIZE),
    'ENABLE_FULL_SHARD': str(FULL_SFT_ENABLE_FULL_SHARD).lower(),
    'OPTIMIZER': FULL_SFT_OPTIMIZER,
    'OUTPUT_DIR': str(FULL_SFT_OUTPUT),
})
full_sft_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/training/train_full_sft.sh')
]
full_sft_started = time.perf_counter()
subprocess.run(full_sft_command, cwd=ROBOTWIN, env=full_sft_env, check=True)
full_sft_elapsed = time.perf_counter() - full_sft_started
print(f'full SFT total: {full_sft_elapsed:.2f}s ({full_sft_elapsed / 60:.2f} min)')
print('checkpoint:', FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}')

### 5.2 Measured full-teacher capacity at global batch 256

The following short sweeps enable the complete depth/video teachers, gradient checkpointing, and FSDP2 full-shard on four and eight W7900 GPUs. Steady time is the mean of optimizer steps 2 and 3. VRAM was sampled once per second across model loading, forward, backward, and the optimizer update; each value is the maximum observed on the indicated GPU. `VRAM usage after building model` is only initialization memory and is not a training peak.

| Platform | Micro batch/GPU | Accumulation | Steady StepTime | Peak VRAM/GPU | Result |
|---|---:|---:|---:|---:|---|
| 4 x W7900 48 GiB | 1 | 64 | 245.580 s | 40.329-40.375 GiB | passed |
| 4 x W7900 48 GiB | 2 | 32 | 136.748 s | 41.337-41.349 GiB | passed |
| 4 x W7900 48 GiB | 4 | 16 | 76.010 s | 43.509-43.521 GiB | passed |
| 4 x W7900 48 GiB | 8 | 8 | 54.031 s | 47.729-47.761 GiB | passed; limited headroom |
| 4 x W7900 48 GiB | 16 | 4 | 41.392 s | 47.890-47.975 GiB | passed; nearly full |
| 4 x W7900 48 GiB | 32 | 2 | - | 47.648-47.956 GiB; only 62-286 MiB free | HIP OOM |
| 8 x W7900 48 GiB | 1 | 32 | 143.665 s | 27.520-27.558 GiB | computed |
| 8 x W7900 48 GiB | 2 | 16 | 80.319 s | 27.818-27.832 GiB | passed |
| 8 x W7900 48 GiB | 4 | 8 | 47.055 s | 29.498-29.523 GiB | passed |
| 8 x W7900 48 GiB | 8 | 4 | 27.108 s | 33.004-33.386 GiB | passed |
| 8 x W7900 48 GiB | 16 | 2 | 24.682 s | 41.344-41.356 GiB | passed |
| 8 x W7900 48 GiB | 32 | 1 | - | 47.878-47.947 GiB; about 0.12-0.14 GiB free | HIP OOM |

Notes: each VRAM range is the minimum-to-maximum peak across the selected cards.


### 5.3 Convert the full-SFT DCP checkpoint

Full SFT has no LoRA adapter to merge. This step gathers the distributed DCP model state and writes a directly loadable Hugging Face checkpoint. Do not use `merge_lora_dcp.py` here.

In [ ]:
FULL_SFT_CHECKPOINT = FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}'
FULL_SFT_MODEL = FULL_SFT_OUTPUT / f'merged_checkpoint/global_step_{FULL_SFT_STEPS}/hf_ckpt'
assert FULL_SFT_CHECKPOINT.exists(), f'Missing checkpoint: {FULL_SFT_CHECKPOINT}'
full_sft_convert_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/convert_full_sft_dcp.py'),
    '--checkpoint', str(FULL_SFT_CHECKPOINT),
    '--training-output', str(FULL_SFT_OUTPUT),
    '--output', str(FULL_SFT_MODEL),
]
convert_started = time.perf_counter()
subprocess.run(full_sft_convert_command, cwd=ROBOTWIN, check=True)
convert_elapsed = time.perf_counter() - convert_started
assert (FULL_SFT_MODEL / 'model.safetensors.index.json').exists(), f'Conversion is incomplete: {FULL_SFT_MODEL}'
print(f'full-SFT conversion total: {convert_elapsed:.2f}s ({convert_elapsed / 60:.2f} min)')
print('full-SFT model:', FULL_SFT_MODEL)

### 5.4 Start the converted full-SFT model on port 13400

In [ ]:
full_sft_server_log = RUNTIME / 'outputs/logs/full_sft_server.log'
full_sft_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(full_sft_server_log), 'False', str(FULL_SFT_MODEL),
]
full_sft_server_handle = full_sft_server_log.open('ab')
FULL_SFT_SERVER_PROCESS = subprocess.Popen(full_sft_server_command, cwd=ROBOTWIN, stdout=full_sft_server_handle, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(600):
    if FULL_SFT_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Full-SFT server exited with code {FULL_SFT_SERVER_PROCESS.returncode}; inspect {full_sft_server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('full-SFT server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Full-SFT server did not become ready; inspect {full_sft_server_log}')


### 5.5 Validate the converted full-SFT model with 10 closed-loop episodes

In [ ]:
full_sft_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
full_sft_eval_elapsed = time.perf_counter() - full_sft_eval_started
print(f'full-SFT evaluation total: {full_sft_eval_elapsed:.2f}s ({full_sft_eval_elapsed / 60:.2f} min)')
print(f'full-SFT evaluation average per episode: {full_sft_eval_elapsed / EVAL_EPISODES:.2f}s')

### 5.6 Optional: run 100 tasks × 10 episodes against the converted full-SFT model

This is also a long 1,000-episode evaluation. Set `FULL_SFT_BENCHMARK_GPU_COUNT` to `4` or `8`; the following code cell passes that value to the unified benchmark script. The measured clean/randomized wall-clock table is in the Robotwin evaluation section of the Guide. The shared script first replaces the full-SFT server with one model server per selected GPU. `--resume` continues the same run by skipping completed task markers. Change `FULL_SFT_BENCHMARK_RUN_NAME` for an independent rerun of the same checkpoint.


In [ ]:
RUN_FULL_SFT_FULL_BENCHMARK = False
FULL_SFT_BENCHMARK_GPU_COUNT = 8  # Notebook default; supported values: 4 or 8
FULL_SFT_BENCHMARK_RUN_NAME = f'full_sft_{FULL_SFT_STEPS}steps_both100x10_{FULL_SFT_BENCHMARK_GPU_COUNT}gpu'

if RUN_FULL_SFT_FULL_BENCHMARK:
    if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
        FULL_SFT_SERVER_PROCESS.wait(timeout=30)
        if 'full_sft_server_handle' in globals() and not full_sft_server_handle.closed:
            full_sft_server_handle.close()
        print('single full-SFT server stopped before the full benchmark')
    benchmark_script = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    subprocess.run([
        PYTHON, str(benchmark_script),
        '--gpu-count', str(FULL_SFT_BENCHMARK_GPU_COUNT),
        '--episodes', '10',
        '--model-path', str(FULL_SFT_MODEL),
        '--expert-check',
        ('--video' if BENCHMARK_VIDEO else '--no-video'),
        '--accept-expert-info-on-failure',
        '--run-name', FULL_SFT_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
        '--resume',
    ], cwd=ROBOTWIN, env=benchmark_env, check=True)
else:
    print('Skipped the long full-SFT 100-task benchmark')


### 5.7 Optional: stop the full-SFT model server

In [ ]:
if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
    FULL_SFT_SERVER_PROCESS.wait(timeout=30)
    if 'full_sft_server_handle' in globals() and not full_sft_server_handle.closed:
        full_sft_server_handle.close()
    print('full-SFT model server stopped')
else:
    print('no full-SFT server started by this notebook kernel')